# SocraTeach Dataset Exploration

**Source:** SocraticLM (NeurIPS 2024) — https://github.com/Ljyustc/SocraticLM  
**File:** `SocraTeach_multi.json`  
**Contents:** 35K multi-round Teacher-Student math tutoring dialogues.

Each entry (keyed by problem ID) contains:
- `question` — the math word problem
- `analysis` — worked solution (ground truth reasoning)
- `answer` — final numeric answer
- `steps` — ideal sequence of guiding sub-questions
- `dialogues` — multiple simulated Teacher-Student conversations for that problem,
  where each turn has `system` (teacher), `user` (student), and `user_type`

 `user_type` — one of six simulated student cognitive states (source: SocraticLM
  paper, Appendix B.1):
  (1) doesn't understand the problem
  (2) doesn't understand the teacher's explanation
  (3) makes a calculation error
  (4) poor knowledge mastery
  (5) weak thirst for learning
  (6) strong in all aspects

## Goal of this notebook
Load the raw file, understand its structure, and compute basic stats
before writing the preprocessing script.

In [5]:
# ---- Imports ----
import json
from collections import Counter

# ---- Load raw dataset ----
DATA_PATH = "../data/raw/socrateach/SocraTeach_multi.json"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Loaded {len(data)} problems")

Loaded 10273 problems


## 1. Structure check
Look at one full record to confirm the shape matches what we expect.

In [6]:
# ---- Peek at a single problem ----
sample_id = list(data.keys())[0]
sample = data[sample_id]

print("Problem ID:", sample_id)
print("Top-level fields:", list(sample.keys()))
print("Number of dialogue variants:", len(sample["dialogues"]))

# Show one full dialogue variant
first_dialogue_id = list(sample["dialogues"].keys())[0]
print("\nSample dialogue:")
for turn in sample["dialogues"][first_dialogue_id]:
    print(turn)

Problem ID: GSM8K_train_0
Top-level fields: ['question', 'analysis', 'answer', 'steps', 'dialogues']
Number of dialogue variants: 5

Sample dialogue:
{'system': 'How many clips did Natalia sell in May?', 'user': 'I\'m confused. Does "half as many" mean the same number as April?', 'user_type': '(1)'}
{'system': 'Not exactly, "half as many" means half the amount. If Natalia sold 48 clips in April, what would be half of 48? That would tell us how many she sold in May. Can you solve that?', 'user': " I'm still not sure. Is it like 48 divided by 2?", 'user_type': '(4)'}
{'system': "Exactly, that's right! If we divide 48 by 2, we get the number of clips Natalia sold in May. What number do you get when you do that?", 'user': 'Oh, I get it now. So 48 divided by 2 is 28.', 'user_type': '(3)'}
{'system': 'Very close! Though there seems to be a little mistake. When you divide 48 by 2, do you get 28 or another number? Try again and see if you get a different result.', 'user': "Oh, I'm sorry. I mus

## 2. Basic statistics
Count problems, dialogues, turns, and message lengths across the whole dataset.

In [7]:
# ---- Compute basic stats across the full dataset ----
num_problems = len(data)
num_dialogues = 0
turn_counts = []
user_type_counter = Counter()
system_lengths = []
user_lengths = []
turns_missing_user = 0
turns_missing_system = 0

for pid, prob in data.items():
    for did, turns in prob["dialogues"].items():
        num_dialogues += 1
        turn_counts.append(len(turns))
        for turn in turns:
            if "system" in turn:
                system_lengths.append(len(turn["system"].split()))
            else:
                turns_missing_system += 1

            if "user" in turn:
                user_lengths.append(len(turn["user"].split()))
                user_type_counter[turn.get("user_type")] += 1
            else:
                turns_missing_user += 1

print(f"Total problems: {num_problems}")
print(f"Total dialogues: {num_dialogues}")
print(f"Avg dialogue variants per problem: {num_dialogues / num_problems:.2f}")
print(f"Avg turns per dialogue: {sum(turn_counts) / len(turn_counts):.2f}")
print(f"Min/Max turns per dialogue: {min(turn_counts)} / {max(turn_counts)}")
print(f"Avg teacher turn length (words): {sum(system_lengths) / len(system_lengths):.1f}")
print(f"Avg student turn length (words): {sum(user_lengths) / len(user_lengths):.1f}")
print(f"\nTurns missing 'system': {turns_missing_system}")
print(f"Turns missing 'user': {turns_missing_user}")

print("\nuser_type distribution:")
for k, v in sorted(user_type_counter.items(), key=lambda x: (x[0] is None, x[0])):
    print(f"  {k}: {v}")

Total problems: 10273
Total dialogues: 35151
Avg dialogue variants per problem: 3.42
Avg turns per dialogue: 5.28
Min/Max turns per dialogue: 3 / 9
Avg teacher turn length (words): 30.3
Avg student turn length (words): 16.3

Turns missing 'system': 0
Turns missing 'user': 35151

user_type distribution:
  (1): 9792
  (2): 8564
  (2)]: 1
  (3): 40845
  (3)]: 14
  (4): 29098
  (4)]: 8
  (5): 9692
  (5)]: 4
  (6): 52410
  (6)]: 23
